

We load train.csv (891 passengers with survival labels) and test.csv (418 passengers to predict).  

RANDOM_STATE = 42 is fixed throughout for reproducibility.


In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import StratifiedKFold
from sklearn.tree import DecisionTreeClassifier

# lock the seed so the shuffles and trees come out the same every time
RANDOM_STATE = 42
# kaggle's public "women live, men die" score, this is the number we want to beat
PUBLIC_BASELINE = 0.74850

# train has the Survived label, test doesn't
# we only predict on test at the very end
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

print(f"Training rows: {len(train)}, Test rows: {len(test)}")
train.head(3)


Training rows: 891, Test rows: 418


,PassengerId,Survived,Pclass,Name,Sex,Age,SibSp,Parch,Ticket,Fare,Cabin,Embarked
0,1,0,3,"Braund, Mr. Owen Harris",male,22.0,1,0,A/5 21171,7.2500,NaN,S
1,2,1,1,"Cumings, Mrs. John Bradley (Florence Briggs Th...",female,38.0,1,0,PC 17599,71.2833,C85,C
2,3,1,3,"Heikkinen, Miss. Laina",female,26.0,0,0,STON/O2. 3101282,7.9250,NaN,S


In [2]:
# gender rule: women get 1 (survived), men get 0 (died)
# astype(int) turns True/False into 1/0 so we can compare it to Survived
gender_train_acc = ((train["Sex"] == "female").astype(int) == train["Survived"]).mean()
print(f"Overall training survival rate: {train['Survived'].mean():.3f}")
print(f"Gender rule training accuracy:  {gender_train_acc:.5f}")
print(f"Kaggle public gender baseline:  {PUBLIC_BASELINE:.5f}")


Overall training survival rate: 0.384
Gender rule training accuracy:  0.78676
Kaggle public gender baseline:  0.74850


Feature Engineering 

Using simple string splitting and column arithmetic:
1. Title: Extracted from `Name` to identify social role and distinguish boys (`Master`) from adult men (`Mr`).
2. FamilySize: `SibSp + Parch + 1`.
3. IsAlone: Binary flag for solo travelers (`FamilySize == 1`).
4. Missing Values: Imputed missing `Age` using median by `Title`, `Fare` with training median, and `Embarked` with training mode (`S`).


In [3]:
def title_from_name(name):
    # names look like "Braund, Mr Owen Harris", so split on the comma then the dot
    title = name.split(",")[1].split(".")[0].strip()
    # fold the rare french titles into the groups they actually belong to
    if title in {"Mme", "Ms"}:
        return "Mrs"
    if title == "Mlle":
        return "Miss"
    # keep the four titles that actually split survival (Master = young boys)
    if title in {"Mr", "Mrs", "Miss", "Master"}:
        return title
    # nobility and other weird titles are too rare to model on their own
    return "Rare"

train["Title"] = train["Name"].map(title_from_name)
test["Title"] = test["Name"].map(title_from_name)

for frame in (train, test):
    # SibSp is siblings/spouses, Parch is parents/children, +1 counts the passenger
    frame["FamilySize"] = frame["SibSp"] + frame["Parch"] + 1
    frame["IsAlone"] = (frame["FamilySize"] == 1).astype(int)

# fill missing values from train only so nothing from test leaks in
age_by_title = train.groupby("Title")["Age"].median()
age_global = train["Age"].median()
fare_median = train["Fare"].median()
embarked_mode = train["Embarked"].mode().iloc[0]

for frame in (train, test):
    # use the title's median age first (a Master is way younger than a Mr), then the overall median
    frame["Age"] = frame["Age"].fillna(frame["Title"].map(age_by_title)).fillna(age_global)
    frame["Fare"] = frame["Fare"].fillna(fare_median)
    frame["Embarked"] = frame["Embarked"].fillna(embarked_mode)

print("Feature engineering complete. Missing values in train:", int(train[["Title", "FamilySize", "Age", "Fare", "Embarked"]].isna().sum().sum()))


Feature engineering complete. Missing values in train: 0


Development History & Earlier Experiments

In initial experiments, we trained standard machine learning classifiers (including HistGradientBoosting and Random Forest). 
* While cross-validation on the training set appeared promising, the boosting model achieved **0.72455** on the Kaggle public leaderboard, falling below the 0.74850 gender baseline.
* Error analysis revealed that complex trees overfit noisy boundary cases: predicting 1st-class cabin adult men alive (who historically died ~65% of the time) and solo 3rd-class women dead (who survived ~62% of the time).
* Consequently, we adopted a conservative modeling strategy: anchor firmly on the historical gender rule, and override it only where empirical evidence across stratified validation folds is clear and deterministic.


Conservative Survival Rules and Decision Tree Comparison 

We examine survival rates for distinct demographic slices:
1. Third-class women in large families ($\ge 5$): In training, only 3 of 27 survived ($11\%$). We predict them dead.
2. Young boys (`Master`) in small families ($\le 4$)**: In training, all 22 survived ($100\%$). We predict them alive.

We compare this rule against a shallow Decision Tree (`max_depth=3`, `min_samples_leaf=10`) using `DictVectorizer` encoding.


In [4]:
def predict_survival(frame):
    # start from the gender rule, then only flip two groups with a really clear pattern
    pred = (frame["Sex"] == "female").astype(int)
    # 3rd class women in families of 5+ basically all died in training
    large_third_class_family = (
        (frame["Sex"] == "female") & (frame["Pclass"] == 3) & (frame["FamilySize"] >= 5)
    )
    # boys titled Master in families of 4 or fewer all survived in training
    small_family_boy = (frame["Title"] == "Master") & (frame["FamilySize"] <= 4)
    # mask only swaps the prediction where the condition is true
    pred = pred.mask(large_third_class_family, 0)
    pred = pred.mask(small_family_boy, 1)
    return pred.astype(int)

# stratified folds keep the survived/died mix about the same in every fold
# later cells reuse this same splitter so the comparisons line up
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
y = train["Survived"]

# same features as the lab 3 tree so the comparison is fair
features = ["Pclass", "Sex", "Title", "FamilySize", "Embarked", "Age", "Fare"]
tree_scores = []
rule_scores = []
gender_scores = []

for train_idx, valid_idx in cv.split(train, y):
    tr = train.iloc[train_idx].copy()
    val = train.iloc[valid_idx].copy()
    actual = y.iloc[valid_idx].to_numpy()

    # score each method only on the held-out fold
    g_pred = (val["Sex"] == "female").astype(int).to_numpy()
    gender_scores.append((g_pred == actual).mean())

    r_pred = predict_survival(val).to_numpy()
    rule_scores.append((r_pred == actual).mean())

    # DictVectorizer turns the mixed text and number columns into a numeric matrix
    # fit it on the training fold only, then transform the validation fold
    vec = DictVectorizer(sparse=False)
    X_tr = vec.fit_transform(tr[features].to_dict("records"))
    X_val = vec.transform(val[features].to_dict("records"))
    # shallow tree: depth 3 and at least 10 rows per leaf so it doesn't overfit as hard
    clf = DecisionTreeClassifier(max_depth=3, min_samples_leaf=10, random_state=RANDOM_STATE)
    clf.fit(X_tr, tr["Survived"])
    t_pred = clf.predict(X_val)
    tree_scores.append((t_pred == actual).mean())

print("--- 5-Fold Cross-Validation Comparison ---")
print(f"Gender Baseline Mean: {np.mean(gender_scores):.4f}")
print(f"Decision Tree Mean:   {np.mean(tree_scores):.4f}")
print(f"Two-Overrides Rule:   {np.mean(rule_scores):.4f}")
print(f"Rule beats gender on every fold: {all(r > g for r, g in zip(rule_scores, gender_scores))}")


--- 5-Fold Cross-Validation Comparison ---
Gender Baseline Mean: 0.7868
Decision Tree Mean:   0.8249
Two-Overrides Rule:   0.8350
Rule beats gender on every fold: True


Group Refinements (Southampton Third-Class Miss)

Third-class unmarried women (`Miss`) boarding at Southampton had an overall survival rate of 32.8% (19 of 58 survived). 
Predicting them dead improves mean cross-validation accuracy to 0.8406 while continuing to beat the gender baseline on every fold.


In [5]:
def predict_survival_v2(frame):
    # same gender starting point as v1, plus one extra override
    pred = (frame["Sex"] == "female").astype(int)
    large_third_class_family = (
        (frame["Sex"] == "female") & (frame["Pclass"] == 3) & (frame["FamilySize"] >= 5)
    )
    # unmarried 3rd class women who boarded at Southampton only survived about a third of the time
    southampton_third_miss = (
        (frame["Title"] == "Miss") & (frame["Pclass"] == 3) & (frame["Embarked"] == "S")
    )
    small_family_boy = (frame["Title"] == "Master") & (frame["FamilySize"] <= 4)
    # flip the two "predict dead" groups first, then the "predict alive" boy override
    pred = pred.mask(large_third_class_family, 0)
    pred = pred.mask(southampton_third_miss, 0)
    pred = pred.mask(small_family_boy, 1)
    return pred.astype(int)

# reuse the same folds as before
# this rule doesn't train on a fold, so we only need the validation slice to score it
v2_scores = []
for _, valid_idx in cv.split(train, y):
    val = train.iloc[valid_idx]
    actual = y.iloc[valid_idx].to_numpy()
    v2_scores.append((predict_survival_v2(val).to_numpy() == actual).mean())

print(f"predict_survival_v2 5-fold mean accuracy: {np.mean(v2_scores):.4f}")
print(f"Beats gender rule on every fold: {all(v > g for v, g in zip(v2_scores, gender_scores))}")


predict_survival_v2 5-fold mean accuracy: 0.8406
Beats gender rule on every fold: True


Ticket-Group Overrides (Model A — Primary Model)

On the Titanic, passengers sharing a `Ticket` traveled and evacuated together:
* For female passengers and boys (`Master`), we check whether other passengers on the same `Ticket` in the labeled training set all survived (`rate == 1`) or all perished (`rate == 0`).
* If unanimous, we override the prediction accordingly.
* Validation & Leakage Considerations: Within the cross-validation loop below, ticket-group survival rates are computed strictly from the training fold (`fold_train`), leaving validation fold labels completely unread during prediction (confirmed via label-inversion testing). Note that while fold-level survival calculation is leakage-free, the structural heuristic thresholds themselves (e.g., family size >= 5, Southampton Miss grouping) were originally discovered through exploratory data analysis on the full training set.

In [6]:
def ticket_group_overrides(target, labeled):
    # start from the v2 rules, then let unanimous ticket-mates override them
    # target is who we're predicting, labeled is who we're allowed to read Survived from
    pred = predict_survival_v2(target).to_numpy().copy()
    # only women and boys get this override, adult men stay predicted as dead
    group = labeled[(labeled["Sex"] == "female") | (labeled["Title"] == "Master")]
    for i, (_, row) in enumerate(target.iterrows()):
        if not (row["Sex"] == "female" or row["Title"] == "Master"):
            continue
        # other labeled passengers on the same ticket, not counting this person
        others = group[(group["Ticket"] == row["Ticket"]) & (group["PassengerId"] != row["PassengerId"])]
        if len(others) == 0:
            continue
        rate = others["Survived"].mean()
        # only override when every ticket-mate lived or every ticket-mate died
        # a mixed group (rate between 0 and 1) leaves the v2 prediction alone
        if rate == 1:
            pred[i] = 1
        elif rate == 0:
            pred[i] = 0
    return pred

combo_scores = []
# out-of-fold preds: each row is predicted without ever seeing its own label
oof_preds = np.zeros(len(train), dtype=int)

for train_idx, valid_idx in cv.split(train, y):
    fold_train, fold_valid = train.iloc[train_idx], train.iloc[valid_idx]
    actual = y.iloc[valid_idx].to_numpy()
    # ticket rates only come from fold_train, so we never peek at the validation labels
    fold_pred = ticket_group_overrides(fold_valid, fold_train)
    oof_preds[valid_idx] = fold_pred
    combo_scores.append((fold_pred == actual).mean())

print("=== Model A Validation Results ===")
print("Fold Scores:     ", [round(s, 4) for s in combo_scores])
print(f"Mean Fold Acc:    {np.mean(combo_scores):.4f}")
print(f"Overall OOF Acc:  {(oof_preds == y.to_numpy()).mean():.4f} ({(oof_preds == y.to_numpy()).sum()} / {len(train)} correct)")
print(f"Beats gender rule every fold: {all(s > g for s, g in zip(combo_scores, gender_scores))}")


=== Model A Validation Results ===
Fold Scores:      [np.float64(0.8715), np.float64(0.8483), np.float64(0.8371), np.float64(0.8539), np.float64(0.8539)]
Mean Fold Acc:    0.8530
Overall OOF Acc:  0.8530 (760 / 891 correct)
Beats gender rule every fold: True


Generate Final Predictions and Format Verification

We apply Model A (`ticket_group_overrides`) fit on the complete labeled training set to predict `test.csv`.  
The resulting predictions are exported to `submission.csv`.


In [7]:
# for the real submission the whole training set is the lookup table
# test has no Survived column, so ticket rates can only come from train
final_pred = ticket_group_overrides(test, train)
submission = pd.DataFrame({
    "PassengerId": test["PassengerId"],
    "Survived": final_pred.astype(int),
})
# index=False keeps the file to the two columns kaggle wants
submission.to_csv("submission.csv", index=False)
print(f"Wrote submission.csv with {len(submission)} rows.")
print(f"Predicted positive survival rate: {submission['Survived'].mean():.4f}")
submission.head(8)


Wrote submission.csv with 418 rows.
Predicted positive survival rate: 0.3325


,PassengerId,Survived
0,892,0
1,893,1
2,894,0
3,895,0
4,896,1
5,897,0
6,898,1
7,899,0


In [8]:
# re-read the file we just wrote so we're checking the csv, not the dataframe in memory
check = pd.read_csv("submission.csv")
assert list(check.columns) == ["PassengerId", "Survived"], "Incorrect column names"
assert len(check) == 418, f"Expected 418 rows, got {len(check)}"
# order has to match test.csv since kaggle scores by row position too
assert check["PassengerId"].tolist() == test["PassengerId"].tolist(), "Passenger IDs do not match test.csv"
assert set(check["Survived"].unique()) <= {0, 1}, "Non-binary predictions detected"
assert check["Survived"].isna().sum() == 0, "Missing values found in predictions"

# if the 80.24% backup file is around, make sure this run didn't drift from it
import os
if os.path.exists("submission_combo_8024_backup.csv"):
    orig_8024 = pd.read_csv("submission_combo_8024_backup.csv")
    diffs = (check["Survived"] != orig_8024["Survived"]).sum()
    assert diffs == 0, f"Discrepancy detected: {diffs} predictions differ from 80.24% baseline!"
    print("Verified: Exactly matches the 0.80239 benchmark backup (0 differences).")

print("ALL SUBMISSION INTEGRITY CHECKS PASSED.")
print("submission.csv has 418 rows, valid PassengerIds, binary predictions, and no missing values.")

Verified: Exactly matches the 0.80239 benchmark backup (0 differences).
ALL SUBMISSION INTEGRITY CHECKS PASSED.
submission.csv has 418 rows, valid PassengerIds, binary predictions, and no missing values.
